# Praktikum Machine Learning: Prediksi Biaya Asuransi

**Nama:** Billy Nanda Alfahrizi  
**Dataset:** Medical Cost Personal Dataset (`insurance.csv`)  
**Jenis masalah:** Regresi

## Tujuan praktikum
Notebook ini digunakan untuk memahami alur kerja machine learning dari awal sampai akhir: memuat data, memeriksa kualitas data, membersihkan data, melakukan eksplorasi data (EDA), menyiapkan fitur, membagi data latih dan data uji, melatih beberapa model regresi, mengevaluasi performa, melakukan validasi silang, lalu menyimpan model terbaik.

**Target prediksi:** `charges`, yaitu biaya asuransi.  
**Catatan:** Nilai `charges` yang tinggi bisa merupakan kondisi nyata, bukan otomatis kesalahan data. Karena itu, outlier diperiksa dan divisualisasikan terlebih dahulu, bukan langsung dihapus atau dibatasi.

## 1. Mengimpor library

Blok kode ini memanggil library yang diperlukan:
- `pandas` dan `numpy` untuk mengolah data.
- `matplotlib` dan `seaborn` untuk membuat grafik.
- Fungsi dari `scikit-learn` untuk membagi data, melakukan preprocessing, melatih model, dan mengevaluasi hasil.
- `joblib` untuk menyimpan model yang sudah dilatih agar bisa digunakan kembali.

Pengaturan `random_state` pada langkah-langkah berikutnya membantu membuat hasil eksperimen lebih konsisten ketika dijalankan ulang.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Memuat dataset

Kode mencoba membaca `insurance.csv`. Jika nama file di folder berbeda dan menggunakan nama `insurance (1).csv`, kode akan mencoba file tersebut juga. Pastikan file CSV berada di folder yang sama dengan notebook.

`df.shape` menunjukkan jumlah baris dan kolom, sedangkan `head()` menampilkan lima baris pertama agar struktur data bisa diperiksa.

In [ ]:
nama_file = "insurance.csv"
if not os.path.exists(nama_file) and os.path.exists("insurance (1).csv"):
    nama_file = "insurance (1).csv"

df = pd.read_csv(nama_file)

print(f"File yang digunakan: {nama_file}")
print("Jumlah baris dan kolom:", df.shape)
display(df.head())

## 3. Memeriksa struktur dan kualitas data

Blok ini memeriksa tipe data, ringkasan statistik, nilai kosong, dan baris duplikat. Pemeriksaan ini penting untuk menentukan apakah data perlu dibersihkan sebelum digunakan untuk melatih model.

- `info()` menampilkan tipe data dan jumlah nilai non-kosong.
- `describe()` merangkum kolom numerik.
- `describe(include="object")` merangkum kolom kategorikal.
- `isnull().sum()` menghitung nilai kosong pada setiap kolom.
- `duplicated().sum()` menghitung baris yang sama persis.

In [ ]:
print("Informasi dataset:")
df.info()

print("\nStatistik deskriptif kolom numerik:")
display(df.describe())

print("\nStatistik deskriptif kolom kategorikal:")
display(df.describe(include="object"))

print("\nMissing values per kolom:")
display(df.isnull().sum().to_frame("jumlah_missing"))

print("Jumlah baris duplikat:", df.duplicated().sum())

## 4. Membersihkan data

Baris duplikat dihapus karena dapat membuat catatan yang sama dihitung lebih dari satu kali. Jika terdapat nilai kosong, kolom kategorikal diisi dengan modus (nilai paling sering muncul), sedangkan kolom numerik diisi dengan median.

Pada dataset ini, kode tetap memeriksa kondisi aktual terlebih dahulu. Jadi, pengisian nilai kosong hanya dilakukan jika memang ditemukan nilai kosong.

In [ ]:
jumlah_awal = len(df)
df = df.drop_duplicates().copy()
print("Jumlah baris duplikat yang dihapus:", jumlah_awal - len(df))

for col in df.columns:
    if df[col].isna().any():
        if df[col].dtype == "object":
            modus = df[col].mode()
            if not modus.empty:
                df[col] = df[col].fillna(modus.iloc[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print("Ukuran data setelah cleaning:", df.shape)
print("\nMissing values setelah cleaning:")
display(df.isnull().sum().to_frame("jumlah_missing"))

## 5. Memahami kolom dan skala pengukuran

Tabel berikut menjelaskan peran setiap kolom. Informasi ini membantu membedakan fitur input dan target, serta menentukan preprocessing yang sesuai.

- `age`, `bmi`, dan `charges` merupakan data numerik kontinu.
- `children` adalah jumlah anak, yaitu data hitungan numerik diskrit.
- `sex`, `smoker`, dan `region` adalah data kategorikal nominal.
- `charges` menjadi target yang ingin diprediksi; kolom lainnya digunakan sebagai fitur.

In [ ]:
deskripsi_kolom = pd.DataFrame({
    "kolom": ["age", "sex", "bmi", "children", "smoker", "region", "charges"],
    "jenis_data": ["Numerik", "Kategorikal", "Numerik", "Numerik diskrit",
                   "Kategorikal", "Kategorikal", "Numerik"],
    "skala/peran": ["Rasio; fitur", "Nominal; fitur", "Rasio; fitur",
                    "Rasio; fitur", "Nominal; fitur", "Nominal; fitur",
                    "Rasio; target"]
})
display(deskripsi_kolom)

## 6. Menganalisis outlier dengan metode IQR

Kode menghitung batas bawah dan batas atas outlier menggunakan metode Interquartile Range (IQR). Nilai di luar rentang tersebut ditandai sebagai calon outlier.

Penting: hasil ini hanya digunakan untuk **mendeteksi dan memahami** data ekstrem. Kolom `charges` tidak langsung dihapus atau diubah karena biaya tinggi bisa berkaitan dengan karakteristik peserta asuransi yang memang nyata.

In [ ]:
kolom_numerik = ["age", "bmi", "children", "charges"]
ringkasan_outlier = []

for col in kolom_numerik:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    batas_bawah = q1 - 1.5 * iqr
    batas_atas = q3 + 1.5 * iqr
    jumlah = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
    ringkasan_outlier.append({
        "kolom": col, "Q1": q1, "Q3": q3, "IQR": iqr,
        "batas_bawah": batas_bawah, "batas_atas": batas_atas,
        "jumlah_calon_outlier": int(jumlah)
    })

display(pd.DataFrame(ringkasan_outlier).round(2))

## 7. Visualisasi outlier menggunakan boxplot

Boxplot membantu melihat sebaran nilai dan titik yang berada di luar whisker. Satu titik yang tampak ekstrem tidak otomatis berarti data salah; interpretasinya perlu mempertimbangkan konteks variabel.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, col in zip(axes.flat, kolom_numerik):
    sns.boxplot(x=df[col], ax=ax)
    ax.set_title(f"Boxplot {col}")
plt.tight_layout()
plt.show()

## 8. Exploratory Data Analysis (EDA)

Bagian ini melihat distribusi biaya asuransi dan hubungan beberapa fitur dengan `charges`. EDA membantu memahami pola data sebelum membangun model, tetapi hubungan yang terlihat pada grafik tidak otomatis membuktikan sebab-akibat.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(df["charges"], kde=True, ax=axes[0])
axes[0].set_title("Distribusi Biaya Asuransi (charges)")
axes[0].set_xlabel("Charges")

sns.scatterplot(data=df, x="age", y="charges", hue="smoker", alpha=0.7, ax=axes[1])
axes[1].set_title("Usia dan Biaya Asuransi berdasarkan Status Merokok")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
sns.boxplot(data=df, x="smoker", y="charges")
plt.title("Biaya Asuransi berdasarkan Status Merokok")
plt.tight_layout()
plt.show()

## 9. Menentukan fitur (X) dan target (y)

`X` berisi kolom yang digunakan untuk membuat prediksi. `y` berisi nilai yang ingin diprediksi, yaitu `charges`. Kolom target tidak boleh dimasukkan ke dalam `X`, karena dapat menyebabkan kebocoran target (target leakage).

Kode juga memeriksa nama kolom yang dibutuhkan agar kesalahan penamaan bisa diketahui sejak awal.

In [ ]:
fitur = ["age", "sex", "bmi", "children", "smoker", "region"]
target = "charges"

kolom_wajib = fitur + [target]
kolom_tidak_ada = [col for col in kolom_wajib if col not in df.columns]
if kolom_tidak_ada:
    raise ValueError(f"Kolom berikut tidak ditemukan dalam dataset: {kolom_tidak_ada}")

X = df[fitur].copy()
y = df[target].copy()

print("Ukuran fitur (X):", X.shape)
print("Ukuran target (y):", y.shape)
display(X.head())

## 10. Membagi data menjadi data latih dan data uji

Data dibagi menjadi 80% data latih dan 20% data uji. Data latih digunakan untuk mempelajari pola, sedangkan data uji dipakai untuk menilai performa pada data yang tidak digunakan saat pelatihan.

`random_state=42` membuat pembagian data dapat diulang. Preprocessing belum dipelajari dari seluruh data; scaler dan encoder akan dipasang dalam pipeline agar hanya di-fit pada data latih ketika model dilatih.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print("Jumlah data latih:", X_train.shape[0])
print("Jumlah data uji:", X_test.shape[0])

## 11. Menyiapkan preprocessing

Fitur numerik distandardisasi menggunakan `StandardScaler`, yang mengubah skala berdasarkan rata-rata dan standar deviasi data latih. Fitur kategorikal diubah menjadi kolom indikator dengan `OneHotEncoder`.

`ColumnTransformer` menerapkan transformasi yang berbeda sesuai jenis kolom. Semua langkah ini akan dimasukkan ke dalam `Pipeline` supaya preprocessing dan model dilatih bersama, sehingga risiko data leakage dapat dikurangi.

In [ ]:
kolom_numerik_fitur = ["age", "bmi", "children"]
kolom_kategorikal = ["sex", "smoker", "region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("numerik", StandardScaler(), kolom_numerik_fitur),
        ("kategorikal", OneHotEncoder(handle_unknown="ignore"), kolom_kategorikal)
    ]
)

print("Preprocessor siap digunakan.")

## 12. Melatih dan membandingkan beberapa model regresi

Beberapa algoritma dibandingkan untuk mengetahui model mana yang paling sesuai:
- `DummyRegressor` menjadi baseline sederhana.
- `LinearRegression` memodelkan hubungan linear.
- `DecisionTreeRegressor` mempelajari pola berbentuk pohon keputusan.
- `RandomForestRegressor` menggabungkan banyak pohon.
- `GradientBoostingRegressor` membangun model secara bertahap untuk memperbaiki kesalahan prediksi.

Setiap model ditempatkan dalam `Pipeline`, lalu dilatih menggunakan `.fit()` hanya pada data latih. Setelah itu, prediksi pada data uji dievaluasi menggunakan MAE, RMSE, dan R².

In [ ]:
model_dasar = {
    "Dummy Regressor": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=RANDOM_STATE)
}

hasil_model = []
model_terlatih = {}

for nama, algoritma in model_dasar.items():
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessor),
        ("model", algoritma)
    ])
    pipeline.fit(X_train, y_train)
    prediksi = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, prediksi)
    rmse = np.sqrt(mean_squared_error(y_test, prediksi))
    r2 = r2_score(y_test, prediksi)

    hasil_model.append({
        "Model": nama, "MAE": mae, "RMSE": rmse, "R2": r2
    })
    model_terlatih[nama] = pipeline

hasil_evaluasi = pd.DataFrame(hasil_model).sort_values("RMSE").reset_index(drop=True)
display(hasil_evaluasi.round(4))

## 13. Memilih model terbaik dan memvisualisasikan hasil

Model terbaik dipilih berdasarkan RMSE terkecil pada data uji. MAE menunjukkan rata-rata besar kesalahan absolut, RMSE memberi penalti lebih besar pada kesalahan yang besar, sedangkan R² menggambarkan proporsi variasi target yang dijelaskan model relatif terhadap baseline rata-rata.

Tidak ada satu metrik yang sempurna untuk semua konteks, jadi hasil sebaiknya dibaca bersama-sama.

In [ ]:
nama_model_terbaik = hasil_evaluasi.iloc[0]["Model"]
model_terbaik = model_terlatih[nama_model_terbaik]

print("Model terbaik berdasarkan RMSE:", nama_model_terbaik)
print(hasil_evaluasi.iloc[0][["MAE", "RMSE", "R2"]].round(4))

plt.figure(figsize=(9, 4))
sns.barplot(data=hasil_evaluasi, x="RMSE", y="Model")
plt.title("Perbandingan RMSE Model (lebih rendah lebih baik)")
plt.tight_layout()
plt.show()

## 14. Validasi silang 5-fold

Validasi silang membagi data latih menjadi lima bagian secara bergantian: sebagian digunakan untuk melatih model dan bagian lainnya untuk validasi. Ini memberikan gambaran performa yang tidak hanya bergantung pada satu pembagian data.

Scoring `neg_root_mean_squared_error` bernilai negatif karena konvensi scoring scikit-learn menganggap skor lebih tinggi selalu lebih baik. Nilainya dikalikan `-1` agar hasil dapat dibaca sebagai RMSE positif.

In [ ]:
cv_scores_neg = cross_val_score(
    model_terbaik, X_train, y_train,
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
cv_rmse = -cv_scores_neg

print("RMSE setiap fold:", np.round(cv_rmse, 2))
print(f"Rata-rata CV RMSE: {cv_rmse.mean():.2f}")
print(f"Standar deviasi CV RMSE: {cv_rmse.std():.2f}")

## 15. Meninjau prediksi pada data uji

Tabel ini membandingkan biaya aktual dengan prediksi model terbaik. Selisih absolut dihitung untuk melihat besar kesalahan pada setiap contoh. Beberapa baris pertama ditampilkan sebagai contoh, bukan sebagai pengganti evaluasi seluruh data uji.

In [ ]:
prediksi_uji = model_terbaik.predict(X_test)
tabel_prediksi = pd.DataFrame({
    "charges_aktual": y_test.to_numpy(),
    "charges_prediksi": prediksi_uji,
    "selisih_absolut": np.abs(y_test.to_numpy() - prediksi_uji)
})
display(tabel_prediksi.head(10).round(2))

## 16. Menyimpan model terbaik

`joblib.dump()` menyimpan pipeline terbaik ke file `insurance_model.joblib`. Karena pipeline mencakup preprocessing dan model, file tersebut dapat menerima data mentah dengan kolom fitur yang sama tanpa harus menjalankan scaler dan encoder secara terpisah.

Simpan file model bersama dokumentasi kolom input yang dibutuhkan. Jangan memuat file model dari sumber yang tidak tepercaya.

In [ ]:
nama_file_model = "insurance_model.joblib"
joblib.dump(model_terbaik, nama_file_model)
print(f"Model tersimpan di: {os.path.abspath(nama_file_model)}")

## 17. Memuat kembali model dan mencoba prediksi

Bagian ini memuat model dari file yang baru disimpan, lalu membuat prediksi pada satu baris data uji. Tujuannya memastikan model dapat digunakan kembali setelah proses penyimpanan.

Prediksi adalah perkiraan berdasarkan pola yang dipelajari dari dataset; hasilnya bukan penawaran resmi atau penentuan biaya asuransi yang sebenarnya.

In [ ]:
model_dimuat = joblib.load(nama_file_model)
contoh_input = X_test.iloc[[0]]
prediksi_contoh = model_dimuat.predict(contoh_input)[0]
biaya_aktual = y_test.iloc[0]

print("Contoh fitur input:")
display(contoh_input)
print(f"Biaya aktual: {biaya_aktual:.2f}")
print(f"Prediksi biaya: {prediksi_contoh:.2f}")
print(f"Selisih absolut: {abs(biaya_aktual - prediksi_contoh):.2f}")

## 18. Kesimpulan otomatis berdasarkan hasil eksperimen

Blok terakhir merangkum model terbaik dan metrik evaluasinya. Kesimpulan ini dibuat dari hasil yang dihitung saat notebook dijalankan, sehingga tidak perlu menuliskan angka evaluasi secara manual.

Saat menulis laporan, tambahkan interpretasi grafik EDA dan jelaskan keterbatasan eksperimen. Hasil pengujian berlaku untuk pembagian data dan dataset yang digunakan pada notebook ini.

In [ ]:
baris_terbaik = hasil_evaluasi.iloc[0]

print("KESIMPULAN")
print(f"Dataset setelah cleaning berisi {len(df)} baris dan {df.shape[1]} kolom.")
print(f"Model terbaik berdasarkan RMSE adalah {baris_terbaik['Model']}.")
print(f"MAE  : {baris_terbaik['MAE']:.2f}")
print(f"RMSE : {baris_terbaik['RMSE']:.2f}")
print(f"R²   : {baris_terbaik['R2']:.4f}")
print(f"Rata-rata CV RMSE 5-fold: {cv_rmse.mean():.2f}")
print("Model disimpan sebagai:", nama_file_model)